In [0]:

from pyspark.sql import functions as F
from pyspark.sql.types import DecimalType

CATALOG = "workspace"

orders = spark.table(
    f"{CATALOG}.bronze.orders"
)

clean_orders = (
    orders
    .withColumn(
        "order_id",
        F.col("order_id").cast("long")
    )
    .withColumn(
        "customer_id",
        F.col("customer_id").cast("long")
    )
    .withColumn(
        "order_date",
        F.to_timestamp("order_date")
    )
    .withColumn(
        "total_amount",
        F.col("total_amount").cast(
            DecimalType(18, 2)
        )
    )
    .withColumn(
        "status",
        F.lower(F.trim(F.col("status")))
    )
)

valid = (
    F.col("order_id").isNotNull()
    & F.col("customer_id").isNotNull()
    & F.col("order_date").isNotNull()
    & F.col("total_amount").isNotNull()
    & (F.col("total_amount") >= 0)
)

invalid_orders = clean_orders.filter(~valid)
clean_orders = (
    clean_orders
    .filter(valid)
    .dropDuplicates(["order_id"])
)

display(clean_orders)


order_id,customer_id,order_date,status,channel,subtotal,discount_amount,shipping_fee,tax_amount,total_amount,currency,_ingested_at,_source_file
1,1,2026-04-18T22:37:25.000Z,delivered,app,14.29,0,7.73,1.14,23.16,USD,2026-11-08T03:29:24.044Z,orders.csv
2,1,2025-06-09T08:05:40.000Z,delivered,marketplace,630.95,0,0,50.48,681.43,USD,2026-11-08T03:29:24.044Z,orders.csv
3,1,2024-11-29T08:01:39.000Z,delivered,app,345.06,62.79,0,22.58,304.85,USD,2026-11-08T03:29:24.044Z,orders.csv
4,1,2024-11-19T20:11:54.000Z,delivered,web,431.38,0,0,34.51,465.89,USD,2026-11-08T03:29:24.044Z,orders.csv
5,2,2025-12-28T04:22:31.000Z,delivered,web,613.4,0,0,49.07,662.47,USD,2026-11-08T03:29:24.044Z,orders.csv
6,2,2024-04-30T16:29:23.000Z,delivered,marketplace,917.21,0,0,73.38,990.59,USD,2026-11-08T03:29:24.044Z,orders.csv
7,2,2026-07-09T21:08:55.000Z,delivered,marketplace,2019.54,0,0,161.56,2181.10,USD,2026-11-08T03:29:24.044Z,orders.csv
8,2,2024-11-02T14:14:42.000Z,delivered,app,438.11,0,0,35.05,473.16,USD,2026-11-08T03:29:24.044Z,orders.csv
9,3,2024-07-20T00:47:06.000Z,delivered,app,2222.19,0,0,177.78,2399.97,USD,2026-11-08T03:29:24.044Z,orders.csv
10,3,2024-01-09T02:18:16.000Z,delivered,app,813.91,0,0,65.11,879.02,USD,2026-11-08T03:29:24.044Z,orders.csv


In [0]:

(
    clean_orders.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        f"{CATALOG}.silver.orders"
    )
)

(
    invalid_orders.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        f"{CATALOG}.silver.invalid_orders"
    )
)

print("Valid orders:", clean_orders.count())
print("Invalid orders:", invalid_orders.count())


Valid orders: 90000
Invalid orders: 0


In [0]:

items = spark.table(
    f"{CATALOG}.bronze.order_items"
)

clean_items = (
    items
    .withColumn(
        "order_item_id",
        F.col("order_item_id").cast("long")
    )
    .withColumn(
        "order_id",
        F.col("order_id").cast("long")
    )
    .withColumn(
        "product_id",
        F.col("product_id").cast("long")
    )
    .withColumn(
        "quantity",
        F.col("quantity").cast("int")
    )
    .withColumn(
        "line_total",
        F.col("line_total").cast(
            DecimalType(18, 2)
        )
    )
    .filter(
        F.col("order_item_id").isNotNull()
        & F.col("order_id").isNotNull()
        & F.col("product_id").isNotNull()
        & (F.col("quantity") > 0)
        & (F.col("line_total") >= 0)
    )
    .dropDuplicates(["order_item_id"])
)

(
    clean_items.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        f"{CATALOG}.silver.order_items"
    )
)


In [0]:

customers = spark.table(
    f"{CATALOG}.bronze.customers"
)

clean_customers = (
    customers
    .withColumn(
        "customer_id",
        F.col("customer_id").cast("long")
    )
    .withColumn(
        "signup_date",
        F.to_date("signup_date")
    )
    .withColumn(
        "country",
        F.trim(F.col("country"))
    )
    .filter(F.col("customer_id").isNotNull())
    .dropDuplicates(["customer_id"])
)

(
    clean_customers.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        f"{CATALOG}.silver.customers"
    )
)

products = spark.table(
    f"{CATALOG}.bronze.products"
)

clean_products = (
    products
    .withColumn(
        "product_id",
        F.col("product_id").cast("long")
    )
    .withColumn(
        "unit_price",
        F.col("unit_price").cast(
            DecimalType(18, 2)
        )
    )
    .filter(
        F.col("product_id").isNotNull()
        & (F.col("unit_price") >= 0)
    )
    .dropDuplicates(["product_id"])
)

(
    clean_products.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        f"{CATALOG}.silver.products"
    )
)


In [0]:

payments = spark.table(
    f"{CATALOG}.bronze.payments"
)

clean_payments = (
    payments
    .withColumn(
        "payment_id",
        F.col("payment_id").cast("long")
    )
    .withColumn(
        "order_id",
        F.col("order_id").cast("long")
    )
    .withColumn(
        "amount",
        F.col("amount").cast(
            DecimalType(18, 2)
        )
    )
    .withColumn(
        "paid_at",
        F.to_timestamp("paid_at")
    )
    .withColumn(
        "status",
        F.lower(F.trim(F.col("status")))
    )
    .filter(
        F.col("payment_id").isNotNull()
        & F.col("order_id").isNotNull()
        & (F.col("amount") >= 0)
    )
    .dropDuplicates(["payment_id"])
)

(
    clean_payments.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        f"{CATALOG}.silver.payments"
    )
)


In [0]:
%sql

SELECT
    COUNT(*) AS total_orders,
    COUNT(DISTINCT order_id) AS distinct_orders,
    SUM(
        CASE WHEN customer_id IS NULL THEN 1 ELSE 0 END
    ) AS missing_customers
FROM workspace.silver.orders;


total_orders,distinct_orders,missing_customers
90000,90000,0
